In [1]:
import time
import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from torch.utils.tensorboard import SummaryWriter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
pima = fetch_openml("pima-indians-diabetes", version=1, as_frame=False)

Xp = StandardScaler().fit_transform(pima["data"]).astype(np.float32)
yp = pima["target"].astype(np.float32).reshape(-1, 1)

Xp_t = torch.tensor(Xp, dtype=torch.float32).to(device)
yp_t = torch.tensor(yp, dtype=torch.float32).to(device)

In [3]:
class NetBin(nn.Module):
    def __init__(self, D, H):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D, H),
            nn.ReLU(),
            nn.Linear(H, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [4]:
model = NetBin(Xp_t.shape[1], 32).to(device)

In [5]:
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.BCELoss()

In [6]:
log_dir = f"logs/pima/{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
writer = SummaryWriter(log_dir=log_dir)

In [7]:
writer.add_graph(model, Xp_t[:1])

In [8]:
epochs = 300
t0 = time.perf_counter()

In [9]:
for epoch in range(1, epochs + 1):
    optimizer.zero_grad()

    out = model(Xp_t)
    loss = criterion(out, yp_t)

    loss.backward()
    optimizer.step()

    pred = (out > 0.5).float()
    acc = (pred == yp_t).float().mean().item()

    # ===== TensorBoard logging =====
    writer.add_scalar("Loss/train", loss.item(), epoch)
    writer.add_scalar("Accuracy/train", acc, epoch)

    # weight histograms
    for name, param in model.named_parameters():
        writer.add_histogram(name, param, epoch)

    if epoch % 50 == 0 or epoch == 1 or epoch == epochs:
        print(
            f"[Pima][Epoch {epoch}/{epochs}] "
            f"loss={loss.item():.6f} acc={acc:.4f} "
            f"elapsed={time.perf_counter() - t0:0.1f}s"
        )

[Pima][Epoch 1/300] loss=0.750342 acc=0.3529 elapsed=0.1s
[Pima][Epoch 50/300] loss=0.679855 acc=0.5651 elapsed=0.4s
[Pima][Epoch 100/300] loss=0.640527 acc=0.6862 elapsed=0.8s
[Pima][Epoch 150/300] loss=0.614084 acc=0.7305 elapsed=1.1s
[Pima][Epoch 200/300] loss=0.593642 acc=0.7448 elapsed=1.4s
[Pima][Epoch 250/300] loss=0.576680 acc=0.7513 elapsed=1.8s
[Pima][Epoch 300/300] loss=0.562180 acc=0.7513 elapsed=2.1s


In [10]:
writer.close()

In [11]:
%load_ext tensorboard
%tensorboard --logdir logs/pima